In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#
import warnings
warnings.filterwarnings('ignore')

#
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

#
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [64]:
df = pd.read_csv("Customer Feedback.csv")

In [65]:
df.head(3)

,CustomerID,Age,Gender,Income,FeedbackScore,ProductUsage,ResponseTime,SupportTickets,Satisfaction_Level
0,C001,25,Male,45000,85.0,12,2.5,1,positive
1,C002,34,FEMALE,62000,92.0,24,1.2,0,Positive
2,C003,41,MAle,55000,45.0,8,5.5,3,Negative


In [66]:
df.shape

(101, 9)

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          101 non-null    object 
 1   Age                 101 non-null    int64  
 2   Gender              100 non-null    object 
 3   Income              101 non-null    object 
 4   FeedbackScore       99 non-null     float64
 5   ProductUsage        101 non-null    int64  
 6   ResponseTime        101 non-null    float64
 7   SupportTickets      101 non-null    int64  
 8   Satisfaction_Level  101 non-null    object 
dtypes: float64(2), int64(3), object(4)
memory usage: 7.2+ KB


In [68]:
df.isnull().sum()

CustomerID            0
Age                   0
Gender                1
Income                0
FeedbackScore         2
ProductUsage          0
ResponseTime          0
SupportTickets        0
Satisfaction_Level    0
dtype: int64

In [69]:
df.duplicated().sum()

0

In [70]:
df['Gender'].value_counts()

Gender
Male      43
Female    41
F          6
FEMALE     3
M          2
MAle       1
female     1
mal        1
FeMale     1
male       1
Name: count, dtype: int64

In [71]:
def fix_gender(txt):
    txt = str(txt).upper()
    if txt[0] == "M":
        return "Male"
    elif txt[0] == "F":
        return "Female"
    else:
        return np.nan

df['Gender'] = df['Gender'].apply(fix_gender)
df['Gender'] = df['Gender'].fillna(method = 'ffill')

In [72]:
df['Gender'].value_counts(dropna = False)

Gender
Female    53
Male      48
Name: count, dtype: int64

In [73]:
df['Income'].value_counts()

Income
43000    4
72000    4
59000    4
45000    3
49000    3
62000    3
57000    3
71000    3
68000    3
74000    3
60000    3
61000    3
58000    3
41000    3
66000    3
47000    3
55000    3
56000    2
75000    2
73000    2
64000    2
54000    2
70000    2
53000    2
50000    2
67000    2
48000    2
51000    2
42000    2
65000    2
46000    2
44000    2
63000    2
69000    2
78000    2
38000    1
40000    1
76000    1
52000    1
77000    1
81000    1
abc      1
82000    1
39000    1
79000    1
80000    1
Name: count, dtype: int64

In [74]:
def fix_income(var):
    try:
        # Try to convert to float (agar integer ya decimal ho)
        int(var)
        return var  # Agar convert ho gaya, to original value return karo
    except:
        return np.nan  # Agar error aaya, to NaN return karo

df['Income'] = df['Income'].apply(fix_income)
df['Income'] = df['Income'].ffill()

In [75]:
df['Income'] = df['Income'].astype(int)

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          101 non-null    object 
 1   Age                 101 non-null    int64  
 2   Gender              101 non-null    object 
 3   Income              101 non-null    int32  
 4   FeedbackScore       99 non-null     float64
 5   ProductUsage        101 non-null    int64  
 6   ResponseTime        101 non-null    float64
 7   SupportTickets      101 non-null    int64  
 8   Satisfaction_Level  101 non-null    object 
dtypes: float64(2), int32(1), int64(3), object(3)
memory usage: 6.8+ KB


In [77]:
df['FeedbackScore'] = df['FeedbackScore'].fillna(method = 'ffill')

In [78]:
df['SupportTickets'].value_counts()

SupportTickets
2    26
1    21
4    18
3    17
0    15
5     4
Name: count, dtype: int64

In [79]:
df['Satisfaction_Level'].value_counts()

Satisfaction_Level
positive                29
neutral                 27
negative                12
Neutral                 10
Negative                 6
Positive                 5
NEG                      3
NEGATIVE                 2
Negative                 1
Neative                  1
Nutreal                  1
Negetive                 1
Nutral                   1
positive  ;duplicate     1
negetive                 1
Name: count, dtype: int64

In [80]:
def fix_satisfaction(txt):
    if pd.isna(txt):  # Handle NaN values
        return np.nan
    
    txt = str(txt).upper().strip()  # Remove extra spaces bhi
    
    # First 3 characters check karo (index 0:3)
    if txt[0:3] == "NEG":  # NEGATIVE, NEG, NEGATIVE etc.
        return "Negative"
    elif txt[0:3] == "POS":  # POSITIVE, POS, POSITIVE etc.
        return 'Positive'
    elif txt[0:3] == "NEU" or txt[0:3] == "NUT":  # NEUTRAL, NEU, NUTRAL etc.
        return 'Neutral'
    else:
        return np.nan

# Apply the function
df['Satisfaction_Level'] = df['Satisfaction_Level'].apply(fix_satisfaction)

# Correct way to fill NaN with backward fill (modern pandas)
df['Satisfaction_Level'] = df['Satisfaction_Level'].fillna(method='bfill')  # Deprecated
# Instead use:
df['Satisfaction_Level'] = df['Satisfaction_Level'].bfill()  # New way

In [81]:
df['Satisfaction_Level'].value_counts(dropna = False)

Satisfaction_Level
Neutral     40
Positive    35
Negative    26
Name: count, dtype: int64

In [82]:
df.head(2)

,CustomerID,Age,Gender,Income,FeedbackScore,ProductUsage,ResponseTime,SupportTickets,Satisfaction_Level
0,C001,25,Male,45000,85.0,12,2.5,1,Positive
1,C002,34,Female,62000,92.0,24,1.2,0,Positive


In [83]:
df = df.drop(columns = ['CustomerID'])

In [84]:
df.head(3)

,Age,Gender,Income,FeedbackScore,ProductUsage,ResponseTime,SupportTickets,Satisfaction_Level
0,25,Male,45000,85.0,12,2.5,1,Positive
1,34,Female,62000,92.0,24,1.2,0,Positive
2,41,Male,55000,45.0,8,5.5,3,Negative


In [85]:
x = df.drop(columns = ['Satisfaction_Level'])
y = df['Satisfaction_Level']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

tf = ColumnTransformer(transformers = [
    ("tf1",MinMaxScaler(), ['Age','FeedbackScore','ProductUsage','ResponseTime']),
    ("tf2", StandardScaler(),['Income']),
    ("tf3", OneHotEncoder(drop = 'first', sparse_output = False), ['Gender'])
],remainder = 'passthrough')

tf

ColumnTransformer(remainder='passthrough',
                  transformers=[('tf1', MinMaxScaler(),
                                 ['Age', 'FeedbackScore', 'ProductUsage',
                                  'ResponseTime']),
                                ('tf2', StandardScaler(), ['Income']),
                                ('tf3',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['Gender'])])

In [86]:
x_train_tf = tf.fit_transform(x_train)
x_test_tf = tf.transform(x_test)

In [91]:
model = RandomForestClassifier()
model.fit(x_train_tf, y_train)
y_pred = model.predict(x_test_tf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy*100:.0f}%")

Accuracy Score: 100%


In [92]:
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

[[ 2  0  0]
 [ 0  8  0]
 [ 0  0 11]]


In [93]:
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         8
           2       1.00      1.00      1.00        11

    accuracy                           1.00        21
   macro avg       1.00      1.00      1.00        21
weighted avg       1.00      1.00      1.00        21



In [143]:
def make_prediction(Age, Gender, Income, FeedbackScore, ProductUsage, ResponseTime, SupportTickets):
    
    x_input = pd.DataFrame({
        "Age": [Age],
        "Gender": [Gender],
        "Income": [Income],
        "FeedbackScore": [FeedbackScore],
        "ProductUsage": [ProductUsage],
        "ResponseTime": [ResponseTime],
        "SupportTickets": [SupportTickets]
    })

    # Transform input using preprocessor
    transform = tf.transform(x_input)
    
    # Make prediction - y_pred already class indices hain
    y_pred = model.predict(transform)
    
    # y_pred already class indices hai, isliye direct use karo
    # y_pred shape check karo
    print(f"y_pred: {y_pred}, type: {type(y_pred)}")
    
    # Convert index to original label
    pred_label = label_encoder.inverse_transform([y_pred])  # y_pred ko list mein wrap karo
    
    return pred_label[0]

In [144]:
Age = 42
Gender = "Male"
Income = 63000
FeedbackScore = 48.0
ProductUsage = 9
ResponseTime = 5.2
SupportTickets = 4

result = make_prediction(Age, Gender, Income, FeedbackScore, ProductUsage, ResponseTime, SupportTickets)
print(result)

y_pred: [0], type: <class 'numpy.ndarray'>
Negative


### Pipline

In [145]:
from sklearn.pipeline import Pipeline

In [146]:
tf = ColumnTransformer(transformers=[
    ("num1", MinMaxScaler(), ['Age','FeedbackScore','ProductUsage','ResponseTime']),
    ("num2", StandardScaler(), ['Income']),
    ("cat", OneHotEncoder(drop='first', sparse_output=False), ['Gender'])
], remainder='passthrough')

In [147]:
pipeline = Pipeline(steps=[
    ("preprocessing", tf),
    ("model", RandomForestClassifier(random_state=42))
])

In [148]:
pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num1', MinMaxScaler(),
                                                  ['Age', 'FeedbackScore',
                                                   'ProductUsage',
                                                   'ResponseTime']),
                                                 ('num2', StandardScaler(),
                                                  ['Income']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['Gender'])])),
                ('model', RandomForestClassifier(random_state=42))])

In [149]:
y_pred = pipeline.predict(x_test)

In [150]:
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.0f}%")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 100%
[[ 2  0  0]
 [ 0  8  0]
 [ 0  0 11]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         8
           2       1.00      1.00      1.00        11

    accuracy                           1.00        21
   macro avg       1.00      1.00      1.00        21
weighted avg       1.00      1.00      1.00        21



In [151]:
result = make_prediction(42, "Male", 63000, 48.0, 9, 5.2, 4)
print(result)

y_pred: [0], type: <class 'numpy.ndarray'>
Negative


In [152]:
pip install fastapi

Defaulting to user installation because normal site-packages is not writeable
     ------------------------------------ 117.0/117.0 kB 402.9 kB/s eta 0:00:00
     -------------------------------------- 74.3/74.3 kB 681.2 kB/s eta 0:00:00
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.1
    Uninstalling typing-inspection-0.4.1:
      Successfully uninstalled typing-inspection-0.4.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [154]:
pip install joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [155]:
import joblib

joblib.dump(pipeline, "model.pkl")
joblib.dump(label_encoder, "label.pkl")

['label.pkl']

In [156]:
from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()

model = joblib.load("model.pkl")
label_encoder = joblib.load("label.pkl")

@app.get("/")
def home():
    return {"message": "Model is running 🚀"}

@app.post("/predict")
def predict(data: dict):
    
    df = pd.DataFrame([data])
    pred = model.predict(df)
    result = label_encoder.inverse_transform(pred)
    
    return {"prediction": result[0]}